# tinychat — Kaggle GPU runbook (ternary vs fp16 emergence frontier)

Runs the GPU-side of the study end to end: **build data → (one-time) judge calibration → 16-run sweep → frontier plot**.
All code, the frozen 4k tokenizer, the 200 frozen prefixes, the rubric and judge prompt come from the repo — nothing here re-derives them.

### Before you start (Kaggle settings, right-hand panel)
- **Accelerator:** GPU **P100** (single 16GB) or **T4** — the harness uses one GPU.
- **Persistence:** set to **Files only** (or *Variables and Files*). This keeps `/kaggle/working` (your `runs/`) across sessions so the sweep **resumes** instead of restarting.
- **Internet:** **On** (needed to pip-install and download TinyStories + the Qwen judge).

### Session strategy (sessions are capped at 12h; quota ~30 GPU-h/week)
The sweep is **idempotent**: every run checkpoints, and finished runs (those with a `DONE` marker) are skipped. If a session times out, just **re-run the sweep cell next session** — it continues from the last checkpoint. Approx GPU time: tiny/small ≪1h each, medium/large ~1–2h each; full 16 runs ≈ 10–20 GPU-h, i.e. ~2 sessions.

## 1. GPU + environment check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
assert torch.cuda.is_available(), "Enable a GPU accelerator in the right-hand panel."

## 2. Get the code + install deps

**Option A (default):** clone from GitHub (`REPO_URL` is preset to `adit-rah/tinychat`). The
cell **always hard-syncs to `origin/main`**, so a stale clone from a previous session can
never run old code. Your data (`/kaggle/working/data`) is separate and untouched.
**Option B:** upload this repo as a Kaggle *Dataset* and set `REPO_DIR = "/kaggle/input/<your-dataset>"`.

> **Do not `pip install -r requirements.txt` here.** That file pins `torch>=2.12` for the
> local dev env; installing it on Kaggle upgrades Kaggle's torch and breaks the preinstalled
> `transformers` (`ImportError: _maybe_view_chunk_cat`). The cell installs only `bitsandbytes`.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/adit-rah/tinychat.git"   # Option A (clone)
REPO_DIR = "/kaggle/working/tinychat"
BRANCH   = "main"
# Option B: REPO_DIR = "/kaggle/input/tinychat"  # uploaded dataset (read-only); skips git

# Always sync to the latest remote so a stale clone from a previous session can never run.
# Repo code lives separately from data (/kaggle/working/data), so re-syncing is safe.
if REPO_DIR.startswith("/kaggle/input"):
    if not os.path.exists("/kaggle/working/tinychat"):
        subprocess.run(["cp", "-r", REPO_DIR, "/kaggle/working/tinychat"], check=True)
    REPO_DIR = "/kaggle/working/tinychat"
elif os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)

print("code at commit:",
      subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip() or "(non-git input)")

# IMPORTANT: do NOT `pip install -r requirements.txt` on Kaggle. It pins torch>=2.12 and would
# upgrade Kaggle's torch (2.10), which breaks the preinstalled transformers with
# "ImportError: cannot import name '_maybe_view_chunk_cat'". Kaggle already ships a compatible
# torch + transformers + datasets + tokenizers stack -- use it as-is. Install ONLY the 4-bit
# judge dep, and (defensively) any of our imports that happen to be missing -- never torch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)

def _ensure(mod):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", mod], check=True)

for m in ("tokenizers", "datasets", "transformers", "accelerate"):
    _ensure(m)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch, transformers
print("repo ready at", REPO_DIR)
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "GPU not visible -- check the accelerator setting"

## 3. Verify the frozen artifacts are present (committed before any run)

In [ ]:
for p in ["artifacts/tokenizer/tokenizer.json", "eval/prefixes.jsonl",
          "eval/rubric.md", "eval/judge_prompt.md"]:
    assert os.path.exists(p), f"MISSING frozen artifact: {p}"
n_prefixes = sum(1 for line in open("eval/prefixes.jsonl") if line.strip())
assert n_prefixes == 200, f"expected 200 prefixes, found {n_prefixes}"
print("frozen artifacts OK | prefixes:", n_prefixes)

## 4. Build the tokenized data (memmaps)
Deterministic and cheap (~a few minutes) — rebuilt each session into `/kaggle/working/data`.

In [ ]:
from datasets import load_dataset
from tinychat.tokenizer import load_tokenizer
from tinychat.data import build_token_memmap

DATA_DIR = "/kaggle/working/data"; os.makedirs(DATA_DIR, exist_ok=True)
RUNS_DIR = "/kaggle/working/runs"; os.makedirs(RUNS_DIR, exist_ok=True)
train_path = os.path.join(DATA_DIR, "train.bin")
val_path   = os.path.join(DATA_DIR, "val.bin")
tok = load_tokenizer("artifacts/tokenizer/tokenizer.json")

if not os.path.exists(train_path):
    n = build_token_memmap((r["text"] for r in load_dataset("roneneldan/TinyStories", split="train")), tok, train_path)
    print("train tokens:", f"{n:,}")
if not os.path.exists(val_path):
    n = build_token_memmap((r["text"] for r in load_dataset("roneneldan/TinyStories", split="validation")), tok, val_path)
    print("val tokens:", f"{n:,}")
print("data ready")

## 5. One-time judge calibration (spec §8 — do this BEFORE the sweep)

Scores three reference sets through the **exact frozen rubric + Qwen2.5-7B-Instruct judge**: real gold continuations (good), an optional TinyStories-33M model (mediocre), and deliberately degenerate text (bad). Writes `eval/calibration.md`.

**You must read the output before sweeping:** good refs should clear ~4.0, garbage should fall well below, and the **intra-judge std must be smaller than the good−bad gap**. If not, the gate is invalid — stop and upgrade the judge (e.g. an Anthropic-API pass) before continuing.

In [ ]:
import importlib.util
from eval.judge import LocalQwenJudge

spec = importlib.util.spec_from_file_location("run_calibration", "scripts/run_calibration.py")
cal = importlib.util.module_from_spec(spec); spec.loader.exec_module(cal)

judge = LocalQwenJudge()  # Qwen/Qwen2.5-7B-Instruct, 4-bit

# OPTIONAL mediocre reference: TinyStories-33M completions. Set USE_33M=True to include.
USE_33M = False
model33m_fn = None
if USE_33M:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    m_id = "roneneldan/TinyStories-33M"
    m_tok = AutoTokenizer.from_pretrained(m_id)
    m = AutoModelForCausalLM.from_pretrained(m_id).to("cuda").eval()
    def model33m_fn(item):
        ids = m_tok(item["prefix"], return_tensors="pt").input_ids.to("cuda")
        out = m.generate(ids, max_new_tokens=200, do_sample=False)
        return m_tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

cal.main(judge=judge, model33m_fn=model33m_fn, repeats=3)
print(open("eval/calibration.md").read())

## 6. Train the sweep (Phase A — training only, idempotent)

Trains every `(tier, precision, seed)` to 500M tokens, skipping any run already marked `DONE`. Re-run this cell across sessions to resume. To cap a session, set `ONLY` to a subset of the matrix.

`micro_rows` controls the grad-accum micro-batch (rows × 512 tokens). Lower it if you hit OOM on the large tier; raise it for speed if memory allows.

In [ ]:
from tinychat.sweep import run_sweep, sweep_matrix

print("matrix:", sweep_matrix())
ONLY = None   # e.g. [("tiny","fp16",0), ("tiny","ternary",0)] to limit this session

trained = run_sweep(
    RUNS_DIR, train_path, val_path,
    only=ONLY, eval_fn=None,                 # train only; judge runs in Phase B
    total_tokens=500_000_000, tokens_per_step=65536, peak_lr=3e-4,
    micro_rows=16, eval_every=250, ckpt_every=500, eval_batches=50,
)
print("trained this session:", [os.path.basename(d) for d in trained])
print("done runs:", sorted(d for d in os.listdir(RUNS_DIR) if os.path.exists(os.path.join(RUNS_DIR, d, 'DONE'))))

## 7. Evaluate every finished run (Phase B — judge once, score all)

Loads the Qwen judge a single time, then for each finished run completes the 200 frozen prefixes and writes `eval.json` (per-prefix scores, mean, 95% CI). Idempotent: runs with an existing `eval.json` are skipped.

In [ ]:
from eval.run_eval import evaluate_run
from eval.judge import LocalQwenJudge

judge = LocalQwenJudge()  # reused across all runs
for tier, prec, seed in sweep_matrix():
    rd = os.path.join(RUNS_DIR, f"{tier}_{prec}_{seed}")
    if not os.path.exists(os.path.join(rd, "DONE")):
        print("skip (not trained yet):", os.path.basename(rd)); continue
    if os.path.exists(os.path.join(rd, "eval.json")):
        print("already evaluated:", os.path.basename(rd)); continue
    r = evaluate_run(rd, judge=judge)
    print(os.path.basename(rd), "-> mean", round(r["mean"], 3),
          "CI [%.3f, %.3f]" % (r["ci_low"], r["ci_high"]))

## 8. Headline frontier plot (coherence vs total bytes)

In [ ]:
from tinychat.plotting import collect_results, plot_frontier
from IPython.display import Image

results = collect_results(RUNS_DIR)
for r in sorted(results, key=lambda z: z["total_bytes"]):
    print(f"{r['tier']:7s} {r['precision']:8s} seed{r['seed']}  "
          f"bytes={r['total_bytes']/1e6:6.2f}MB  ppl={r['val_ppl']}  judge={r['judge_mean']}")

out_png = "/kaggle/working/frontier.png"
headline = plot_frontier(results, out_png)
print("\nHEADLINE:", headline)
Image(out_png)

## 9. Save outputs + write up

- **Persist / resume:** with Persistence = *Files only*, `runs/` in `/kaggle/working` survives to the next session. Use **Save Version (Commit)** to snapshot outputs; you can also add this notebook's output as an input dataset to a fresh session.
- **Download:** `frontier.png`, each `runs/*/metrics.csv`, `runs/*/eval.json`, and `eval/calibration.md` are the deliverables. Copy them out of `/kaggle/working` (right-hand *Output* panel) or commit a version.
- **Writeup:** fill in `docs/writeup.md` (the 1–2 page finding): where ternary wins on bytes / where it crosses, the smallest-capable headline (or *tiers indistinguishable* if CIs straddle 4.0), the compute-axis caveat (ternary costs **more** to train), and the embedding-dominance lesson.

```python
import shutil, glob
os.makedirs('/kaggle/working/deliverables', exist_ok=True)
shutil.copy('/kaggle/working/frontier.png', '/kaggle/working/deliverables/')
shutil.copy('eval/calibration.md', '/kaggle/working/deliverables/')
for f in glob.glob(os.path.join(RUNS_DIR,'*','metrics.csv')) + glob.glob(os.path.join(RUNS_DIR,'*','eval.json')):
    run = os.path.basename(os.path.dirname(f))
    shutil.copy(f, f'/kaggle/working/deliverables/{run}_{os.path.basename(f)}')
```